# 04 — ECG Signal Forecasting: Fixed & Complete Pipeline
**PTB-XL 12-Lead ECG — Frankfurt UAS · HIS Masters Project**

## What was wrong and what is fixed

| Problem | Old approach | Fixed approach |
|---|---|---|
| Dataset size | 200 records → 635 train windows | ALL PTB-XL records → 100K+ windows |
| Decoder | `pool → Linear(1200)` — collapse to mean | Seq2Seq LSTM decoder / temporal conv |
| Transformer pooling | `mean(dim=1)` — destroys phase | Last token output — preserves position |
| Horizon difficulty | 100 steps × 12 leads = 1200 outputs | 50 steps × 12 leads = 600 outputs |
| Models | LSTM, CNN-LSTM, Transformer | + TCN + WaveNet = 5 models |
| Metrics | MAE, RMSE, MAPE only | + Pearson r + persistence baseline |

## Task formulation
```
Given:   X_in  ∈ R^(500 × 12)  — 5 seconds past ECG, all 12 leads
Predict: y_hat ∈ R^(50  × 12)  — next 0.5 seconds (reduced for learnability)
Loss:    MSE (direct, no Huber smoothing hiding peaks)
```

### Cell 1 — Install & Import

In [1]:
!pip install wfdb pandas numpy matplotlib scikit-learn scipy tqdm torch --quiet

import os, ast, pickle, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.signal import butter, filtfilt
from scipy.stats import pearsonr
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("✅ Imports ready")

Device : cpu
✅ Imports ready


### Cell 2 — Stream ALL PTB-XL Records Online

In [2]:
# ============================================================
# STREAM ALL PTB-XL RECORDS FROM PHYSIONET
# This is the critical fix — 200 records was far too small.
# We stream all ~21,000 records. Each record = 10s @ 100Hz.
# We take a large random subset for development (e.g. 2000),
# but use ALL for final training.
# ============================================================
import wfdb

PHYSIONET_DB = 'ptb-xl/1.0.3'
FS           = 100
INPUT_LEN    = 500    # 5 seconds
HORIZON      = 50     # 0.5 seconds — reduced for learnability
STRIDE       = 50     # dense windowing → more training samples
LEAD_NAMES   = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
N_LEADS      = 12

# Load metadata
print("Loading metadata from PhysioNet...")
df = pd.read_csv(
    'https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv',
    index_col='ecg_id'
)
df.scp_codes = df.scp_codes.apply(ast.literal_eval)
print(f"Total records available: {len(df):,}")

# ── Preprocessing functions ───────────────────────────────────
def bandpass(sig, lo=0.5, hi=40.0, fs=100, order=4):
    nyq  = 0.5 * fs
    b, a = butter(order, [lo/nyq, hi/nyq], btype='band')
    return filtfilt(b, a, sig, axis=0)

def zscore(sig):
    mu  = sig.mean(axis=0, keepdims=True)
    std = sig.std(axis=0,  keepdims=True)
    std[std < 1e-8] = 1.0
    return (sig - mu) / std

def preprocess_record(sig):
    """Full preprocessing pipeline on one record."""
    # 1. NaN interpolation
    df_s = pd.DataFrame(sig)
    if df_s.isna().any().any():
        sig = df_s.interpolate(method='linear', limit_direction='both').values
    # 2. Outlier clip ±3σ
    mu, sd = sig.mean(axis=0), sig.std(axis=0)
    sig    = np.clip(sig, mu - 3*sd, mu + 3*sd)
    # 3. Bandpass 0.5–40 Hz
    sig = bandpass(sig)
    # 4. Per-lead Z-score
    sig = zscore(sig)
    return sig.astype(np.float32)

# ── Stream records in batches ─────────────────────────────────
# Use ALL records — set N_RECORDS=None for full dataset
# For Colab with limited RAM, 3000 gives ~80K windows
N_RECORDS = None   # ← set to None to use all 21,799

record_list = df['filename_lr'].tolist()
if N_RECORDS:
    # Stratified sample — take N_RECORDS preserving fold distribution
    sampled = df.groupby('strat_fold').apply(
        lambda x: x.sample(min(len(x), N_RECORDS // 10), random_state=42)
    ).reset_index(level=0, drop=True)
    record_list = sampled['filename_lr'].tolist()
    meta_subset = sampled
else:
    meta_subset = df

print(f"\nStreaming {len(record_list):,} records from PhysioNet...")
print("This will take a few minutes. Progress:")

signals_raw = []
folds       = []
failed      = 0

for idx, (raw_path, fold) in enumerate(
        zip(meta_subset['filename_lr'], meta_subset['strat_fold'])):
    folder   = '/'.join(raw_path.split('/')[:-1])
    rec_name = raw_path.split('/')[-1]
    pn_dir   = f"{PHYSIONET_DB}/{folder}"
    try:
        sig, _ = wfdb.rdsamp(rec_name, pn_dir=pn_dir)
        sig    = preprocess_record(sig)   # (1000, 12)
        signals_raw.append(sig)
        folds.append(int(fold))
    except Exception:
        failed += 1

    if (idx + 1) % 500 == 0:
        print(f"  {idx+1:>6}/{len(record_list)}  loaded={len(signals_raw)}  failed={failed}")

print(f"\n✅ Loaded: {len(signals_raw):,} records  |  Failed: {failed}")
print(f"   Memory estimate: {len(signals_raw) * 1000 * 12 * 4 / 1e6:.0f} MB")

Loading metadata from PhysioNet...
Total records available: 21,799

Streaming 21,799 records from PhysioNet...
This will take a few minutes. Progress:
     500/21799  loaded=500  failed=0
    1000/21799  loaded=1000  failed=0
    1500/21799  loaded=1500  failed=0
    2000/21799  loaded=2000  failed=0
    2500/21799  loaded=2500  failed=0
    3000/21799  loaded=3000  failed=0
    3500/21799  loaded=3500  failed=0
    4000/21799  loaded=4000  failed=0
    4500/21799  loaded=4500  failed=0
    5000/21799  loaded=5000  failed=0
    5500/21799  loaded=5500  failed=0
    6000/21799  loaded=6000  failed=0
    6500/21799  loaded=6500  failed=0
    7000/21799  loaded=7000  failed=0
    7500/21799  loaded=7500  failed=0
    8000/21799  loaded=8000  failed=0
    8500/21799  loaded=8500  failed=0
    9000/21799  loaded=9000  failed=0
    9500/21799  loaded=9500  failed=0
   10000/21799  loaded=10000  failed=0
   10500/21799  loaded=10500  failed=0
   11000/21799  loaded=11000  failed=0
   11500/21

### Cell 3 — Build Forecasting Windows (Dense Striding)

In [3]:
# ============================================================
# BUILD FORECAST WINDOWS WITH DENSE STRIDE
# Dense stride = HORIZON (50) → many more windows per record
# A 1000-sample record yields (1000-500-50)/50 + 1 = 9 windows
# 10,000 records → ~90,000 training windows
# ============================================================
def build_windows(signals, folds_list, input_len, horizon, stride):
    X_list, y_list, fold_list = [], [], []
    total = input_len + horizon

    for sig, fold in zip(signals, folds_list):
        n = sig.shape[0]
        start = 0
        while start + total <= n:
            X_list.append(sig[start : start + input_len])
            y_list.append(sig[start + input_len : start + total])
            fold_list.append(fold)
            start += stride

    return (np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.float32),
            np.array(fold_list, dtype=np.int32))

X_all, y_all, fold_all = build_windows(signals_raw, folds, INPUT_LEN, HORIZON, STRIDE)

# Patient-safe split
train_m = fold_all <= 8
val_m   = fold_all == 9
test_m  = fold_all == 10

X_train, y_train = X_all[train_m], y_all[train_m]
X_val,   y_val   = X_all[val_m],   y_all[val_m]
X_test,  y_test  = X_all[test_m],  y_all[test_m]

total = len(X_all)
print("=" * 60)
print("  FORECASTING WINDOWS BUILT")
print("=" * 60)
print(f"  INPUT_LEN : {INPUT_LEN} samples ({INPUT_LEN/FS:.1f}s)")
print(f"  HORIZON   : {HORIZON}  samples ({HORIZON/FS:.1f}s)  ← reduced from 100")
print(f"  STRIDE    : {STRIDE}   samples (dense windowing)")
print(f"  X shape   : {X_train.shape}  → (windows, input_len, leads)")
print(f"  y shape   : {y_train.shape}  → (windows, horizon, leads)")
print()
print(f"  Train (f1-8) : {len(X_train):>8,} windows  ({100*len(X_train)/total:.1f}%)")
print(f"  Val   (f9)   : {len(X_val):>8,} windows  ({100*len(X_val)/total:.1f}%)")
print(f"  Test  (f10)  : {len(X_test):>8,} windows  ({100*len(X_test)/total:.1f}%)")
print(f"  Total        : {total:>8,} windows")
print()
print(f"  ✅ {len(X_train):,} train windows (was 635 — this is the fix)")

  FORECASTING WINDOWS BUILT
  INPUT_LEN : 500 samples (5.0s)
  HORIZON   : 50  samples (0.5s)  ← reduced from 100
  STRIDE    : 50   samples (dense windowing)
  X shape   : (174180, 500, 12)  → (windows, input_len, leads)
  y shape   : (174180, 50, 12)  → (windows, horizon, leads)

  Train (f1-8) :  174,180 windows  (79.9%)
  Val   (f9)   :   21,830 windows  (10.0%)
  Test  (f10)  :   21,970 windows  (10.1%)
  Total        :  217,980 windows

  ✅ 174,180 train windows (was 635 — this is the fix)


### Cell 4 — Dataset & DataLoaders

In [4]:
class ECGDataset(Dataset):
    def __init__(self, X, y, channel_first=False):
        self.X  = torch.from_numpy(X)
        self.y  = torch.from_numpy(y)
        self.cf = channel_first

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]               # (500, 12)
        if self.cf: x = x.permute(1,0)  # (12, 500) for CNN models
        return x, self.y[idx]

def make_loaders(cf=False, bs_tr=128, bs_ev=256):
    kw = dict(num_workers=0, pin_memory=(DEVICE.type=='cuda'))
    tr = DataLoader(ECGDataset(X_train,y_train,cf), bs_tr,
                    shuffle=True, drop_last=True, **kw)
    vl = DataLoader(ECGDataset(X_val,  y_val,  cf), bs_ev, **kw)
    te = DataLoader(ECGDataset(X_test, y_test, cf), bs_ev, **kw)
    return tr, vl, te

# Time-first  (LSTM, Transformer, TCN, WaveNet)
lstm_tr, lstm_vl, lstm_te = make_loaders(cf=False)
# Channel-first  (CNN-LSTM)
cnn_tr, cnn_vl, cnn_te   = make_loaders(cf=True)

xb, yb = next(iter(lstm_tr))
print(f"Time-first batch    : x={xb.shape}  y={yb.shape}")
xb2, _ = next(iter(cnn_tr))
print(f"Channel-first batch : x={xb2.shape}")
print(f"Train batches: {len(lstm_tr)}  Val: {len(lstm_vl)}  Test: {len(lstm_te)}")
print("✅ DataLoaders ready")

Time-first batch    : x=torch.Size([128, 500, 12])  y=torch.Size([128, 50, 12])
Channel-first batch : x=torch.Size([128, 12, 500])
Train batches: 1360  Val: 86  Test: 86
✅ DataLoaders ready


### Cell 5 — Naive Persistence Baseline

In [5]:
# ============================================================
# PERSISTENCE BASELINE
# The simplest possible forecast: repeat the last observed sample.
# Every model must beat this to be considered useful.
# ============================================================
def persistence_forecast(X):
    """Repeat last sample of input for HORIZON steps."""
    last = X[:, -1:, :]           # (N, 1, 12)
    return np.repeat(last, HORIZON, axis=1)  # (N, HORIZON, 12)

y_persist = persistence_forecast(X_test)
persist_mae  = mean_absolute_error(y_test.reshape(-1), y_persist.reshape(-1))
persist_rmse = np.sqrt(mean_squared_error(y_test.reshape(-1), y_persist.reshape(-1)))
persist_corr = np.mean([
    pearsonr(y_test[:,t,0], y_persist[:,t,0])[0]
    for t in range(HORIZON)
])

print("=" * 50)
print("  PERSISTENCE BASELINE (must beat this)")
print("=" * 50)
print(f"  MAE  : {persist_mae:.4f}")
print(f"  RMSE : {persist_rmse:.4f}")
print(f"  Corr : {persist_corr:.4f}")
print()
print("  Any trained model with lower RMSE and higher")
print("  correlation than this is genuinely learning.")

  PERSISTENCE BASELINE (must beat this)
  MAE  : 0.8954
  RMSE : 1.4037
  Corr : -0.0061

  Any trained model with lower RMSE and higher
  correlation than this is genuinely learning.


### Cell 6 — Model A: Seq2Seq LSTM (Fixed)

**Key fix:** replaced `pool → Linear(1200)` with a proper **LSTM decoder** that generates
one timestep at a time. This preserves temporal structure in the output.

In [6]:
# ============================================================
# MODEL A: SEQ2SEQ LSTM — ENCODER + LSTM DECODER
#
# FIX: The decoder now generates the forecast autoregressively
# using an LSTM decoder cell, not a flat linear projection.
# This is the standard seq2seq architecture for time series.
#
# Encoder: BiLSTM reads 500 input steps → context (h, c)
# Decoder: Unidirectional LSTM generates 50 output steps
#          using the encoder's final state as initialisation
# ============================================================
class Seq2SeqLSTM(nn.Module):
    """
    Seq2Seq LSTM forecaster with LSTM decoder.

    Encoder: Bidirectional LSTM (hidden=128, 2 layers)
             processes all 500 input timesteps.
    Decoder: Unidirectional LSTM (hidden=256)
             generates one output step at a time,
             conditioned on the encoder's final state.

    Input  : (B, 500, 12)
    Output : (B, HORIZON, 12) — continuous mV forecast
    """
    def __init__(self, n_leads=12, hidden=128, n_layers=2,
                 dropout=0.2, horizon=50):
        super().__init__()
        self.horizon  = horizon
        self.n_leads  = n_leads
        self.hidden   = hidden

        # Encoder — bidirectional
        self.encoder = nn.LSTM(
            n_leads, hidden, n_layers,
            batch_first=True, dropout=dropout if n_layers>1 else 0.0,
            bidirectional=True
        )

        # Bridge: map bidir encoder state → decoder state
        # bidir encoder final hidden: (2*n_layers, B, hidden)
        # we need: (n_layers, B, hidden*2) for decoder
        self.bridge_h = nn.Linear(hidden * 2, hidden * 2)
        self.bridge_c = nn.Linear(hidden * 2, hidden * 2)

        # Decoder — unidirectional, hidden = hidden*2 to match bidir
        self.decoder = nn.LSTM(
            n_leads, hidden * 2, num_layers=1,
            batch_first=True, dropout=0.0
        )

        # Output projection: hidden → n_leads per step
        self.out_proj = nn.Sequential(
            nn.Linear(hidden * 2, 64),
            nn.GELU(),
            nn.Linear(64, n_leads)
        )

    def forward(self, x, teacher_forcing_ratio=0.0, target=None):
        B = x.size(0)

        # ── Encoder ───────────────────────────────────────────
        _, (h_n, c_n) = self.encoder(x)
        # h_n: (2*n_layers, B, hidden)
        # Reshape: take last layer from each direction, concatenate
        # Forward last layer: h_n[-2], Backward last layer: h_n[-1]
        h_fwd = h_n[-2]   # (B, hidden)
        h_bwd = h_n[-1]   # (B, hidden)
        c_fwd = c_n[-2]
        c_bwd = c_n[-1]

        h_enc = torch.cat([h_fwd, h_bwd], dim=1)  # (B, hidden*2)
        c_enc = torch.cat([c_fwd, c_bwd], dim=1)

        # Bridge through linear layer + tanh
        h_dec = torch.tanh(self.bridge_h(h_enc)).unsqueeze(0)  # (1,B,hidden*2)
        c_dec = torch.tanh(self.bridge_c(c_enc)).unsqueeze(0)

        # ── Decoder — one step at a time ──────────────────────
        # Start token: last observed sample from input
        dec_input = x[:, -1:, :]    # (B, 1, 12)
        outputs   = []

        for t in range(self.horizon):
            dec_out, (h_dec, c_dec) = self.decoder(dec_input, (h_dec, c_dec))
            # dec_out: (B, 1, hidden*2)
            pred = self.out_proj(dec_out)    # (B, 1, n_leads)
            outputs.append(pred)

            # Teacher forcing during training (when target provided)
            if self.training and target is not None and torch.rand(1).item() < teacher_forcing_ratio:
                dec_input = target[:, t:t+1, :]
            else:
                dec_input = pred   # use own prediction as next input

        return torch.cat(outputs, dim=1)   # (B, HORIZON, 12)


lstm_model = Seq2SeqLSTM(
    n_leads=N_LEADS, hidden=128, n_layers=2,
    dropout=0.2, horizon=HORIZON
).to(DEVICE)

n_lstm = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)

with torch.no_grad():
    d = torch.randn(4, INPUT_LEN, N_LEADS).to(DEVICE)
    o = lstm_model(d)
    print(f"Seq2Seq LSTM: {d.shape} → {o.shape}  ✅")
print(f"Parameters  : {n_lstm:,}")

Seq2Seq LSTM: torch.Size([4, 500, 12]) → torch.Size([4, 50, 12])  ✅
Parameters  : 965,964


### Cell 7 — Model B: CNN-LSTM (Fixed Decoder)

In [7]:
# ============================================================
# MODEL B: CNN-LSTM — Fixed with Seq2Seq decoder
# CNN compresses 500 → 62 for feature extraction,
# then seq2seq LSTM decoder generates output step by step.
# ============================================================
class CNNSeq2SeqLSTM(nn.Module):
    """
    CNN feature extractor + Seq2Seq LSTM decoder.

    CNN:     (B, 12, 500) → (B, 128, 62) local features
    Encoder: LSTM over 62 compressed timesteps
    Decoder: LSTM generates HORIZON output steps
    """
    def __init__(self, n_leads=12, horizon=50, dropout=0.2, cnn_ch=128, lstm_h=128):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads
        self.lstm_h  = lstm_h

        # CNN: (B, 12, 500) → (B, 128, 62)
        self.cnn = nn.Sequential(
            nn.Conv1d(n_leads, 32, 7, padding=3, bias=False),
            nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2, bias=False),
            nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64, cnn_ch, 3, padding=1, bias=False),
            nn.BatchNorm1d(cnn_ch), nn.GELU(), nn.MaxPool1d(2),
        )

        self.encoder = nn.LSTM(
            cnn_ch, lstm_h, 2, batch_first=True,
            dropout=dropout, bidirectional=True
        )

        self.bridge_h = nn.Linear(lstm_h * 2, lstm_h * 2)
        self.bridge_c = nn.Linear(lstm_h * 2, lstm_h * 2)

        self.decoder = nn.LSTM(
            n_leads, lstm_h * 2, num_layers=1, batch_first=True
        )

        self.out_proj = nn.Sequential(
            nn.Linear(lstm_h * 2, 64),
            nn.GELU(),
            nn.Linear(64, n_leads)
        )

    def forward(self, x, teacher_forcing_ratio=0.0, target=None):
        # x: (B, 12, 500)  channel-first
        feats = self.cnn(x).permute(0, 2, 1)     # (B, 62, 128)

        _, (h_n, c_n) = self.encoder(feats)
        h_enc = torch.cat([h_n[-2], h_n[-1]], 1)
        c_enc = torch.cat([c_n[-2], c_n[-1]], 1)
        h_dec = torch.tanh(self.bridge_h(h_enc)).unsqueeze(0)
        c_dec = torch.tanh(self.bridge_c(c_enc)).unsqueeze(0)

        # Need original last input sample for start token
        # x is channel-first: (B, 12, 500) → last step: (B, 12) → (B, 1, 12)
        dec_input = x[:, :, -1].unsqueeze(1)   # (B, 1, 12)
        outputs   = []

        for t in range(self.horizon):
            dec_out, (h_dec, c_dec) = self.decoder(dec_input, (h_dec, c_dec))
            pred = self.out_proj(dec_out)
            outputs.append(pred)
            if self.training and target is not None and torch.rand(1).item() < teacher_forcing_ratio:
                dec_input = target[:, t:t+1, :]
            else:
                dec_input = pred

        return torch.cat(outputs, dim=1)   # (B, HORIZON, 12)


cnn_lstm_model = CNNSeq2SeqLSTM(
    n_leads=N_LEADS, horizon=HORIZON, dropout=0.2
).to(DEVICE)

n_cnn = sum(p.numel() for p in cnn_lstm_model.parameters() if p.requires_grad)

with torch.no_grad():
    d = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)
    o = cnn_lstm_model(d)
    print(f"CNN-LSTM: {d.shape} → {o.shape}  ✅")
print(f"Params  : {n_cnn:,}")

CNN-LSTM: torch.Size([4, 12, 500]) → torch.Size([4, 50, 12])  ✅
Params  : 1,122,700


### Cell 8 — Model C: Transformer (Fixed — Last Token, Not Mean Pool)

In [8]:
# ============================================================
# MODEL C: TRANSFORMER FORECASTER — FIXED
#
# KEY FIX: Replace mean pooling with LAST TOKEN output.
# Mean pooling:  destroys phase information — model can't know
#                where in the cardiac cycle the input ends.
# Last token:    preserves phase — the final encoded position
#                represents the most recent state of the signal,
#                exactly what a forecaster needs to continue from.
#
# Also: output head generates sequence via transposed conv,
# not a flat Linear(1200) projection.
# ============================================================
class TransformerForecaster(nn.Module):
    """
    Transformer encoder with last-token readout for ECG forecasting.

    Input  : (B, 500, 12)
    Output : (B, HORIZON, 12)

    Architecture:
    1. Linear input projection: 12 → d_model
    2. Sinusoidal positional encoding
    3. Transformer encoder (4 layers, 8 heads, pre-LN)
    4. Last token readout — preserves phase position
    5. Temporal upsampling decoder: produces HORIZON steps
    """
    def __init__(self, n_leads=12, d_model=128, nhead=8,
                 n_layers=4, d_ff=256, dropout=0.1, horizon=50):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads
        self.d_model = d_model

        self.input_proj = nn.Linear(n_leads, d_model)

        # Positional encoding
        pe  = torch.zeros(1000, d_model)
        pos = torch.arange(1000).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

        self.pe_drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_ff, dropout=dropout,
            activation='gelu', batch_first=True,
            norm_first=True   # Pre-LN: more stable for regression
        )
        self.encoder = nn.TransformerEncoder(
            enc_layer, num_layers=n_layers,
            norm=nn.LayerNorm(d_model)
        )

        # Temporal decoder: expand last token → HORIZON steps
        # Uses transposed convolution for smooth temporal upsampling
        self.temporal_expand = nn.Sequential(
            nn.Linear(d_model, horizon * d_model // 4),
            nn.GELU(),
        )
        # Reshape to (B, d_model//4, horizon) then conv
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(d_model // 4, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(64, n_leads, kernel_size=3, padding=1),
        )

    def forward(self, x):
        B, T, _ = x.shape
        x = self.pe_drop(
            self.input_proj(x) + self.pe[:, :T]
        )                                         # (B, 500, d_model)
        enc = self.encoder(x)                     # (B, 500, d_model)

        # Last token — represents end-of-input phase position
        last = enc[:, -1, :]                      # (B, d_model)

        # Expand to temporal sequence
        out = self.temporal_expand(last)           # (B, horizon * d_model//4)
        out = out.view(B, self.d_model // 4, self.horizon)  # (B, d_model//4, horizon)
        out = self.temporal_conv(out)              # (B, n_leads, horizon)
        return out.permute(0, 2, 1)               # (B, horizon, n_leads)


transformer_model = TransformerForecaster(
    n_leads=N_LEADS, d_model=128, nhead=8,
    n_layers=4, d_ff=256, dropout=0.1, horizon=HORIZON
).to(DEVICE)

n_trans = sum(p.numel() for p in transformer_model.parameters() if p.requires_grad)

with torch.no_grad():
    d = torch.randn(4, INPUT_LEN, N_LEADS).to(DEVICE)
    o = transformer_model(d)
    print(f"Transformer: {d.shape} → {o.shape}  ✅")
print(f"Params     : {n_trans:,}")

Transformer: torch.Size([4, 500, 12]) → torch.Size([4, 50, 12])  ✅
Params     : 746,764


### Cell 9 — Model D: TCN (Temporal Convolutional Network)

In [9]:
# ============================================================
# MODEL D: TEMPORAL CONVOLUTIONAL NETWORK (TCN)
#
# TCN uses causal dilated convolutions with residual connections.
# Key properties:
# - Causal: output at t depends only on inputs ≤ t (no future leak)
# - Dilated: exponentially growing receptive field
#   Dilation 1,2,4,8,16,32,64,128,256 → covers 512 samples ≥ 500 ✅
# - Parallel: entire sequence processed simultaneously (no BPTT)
# - Stable gradients: residual connections throughout
#
# This is often the most practical model for ECG — fast, stable,
# interpretable receptive field.
# ============================================================
class CausalConv1d(nn.Module):
    """Causal (left-padded) dilated convolution."""
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_ch, out_ch, kernel_size,
            dilation=dilation, padding=0, bias=False
        )

    def forward(self, x):
        # Left-pad to maintain causality
        x = F.pad(x, (self.padding, 0))
        return self.conv(x)


class TCNBlock(nn.Module):
    """
    TCN residual block:
    CausalConv → WeightNorm → GELU → Dropout → CausalConv → WeightNorm → GELU
    + residual connection (1×1 conv if channels differ)
    """
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        self.conv1 = nn.utils.weight_norm(CausalConv1d(in_ch, out_ch, kernel_size, dilation))
        self.conv2 = nn.utils.weight_norm(CausalConv1d(out_ch, out_ch, kernel_size, dilation))
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(dropout)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        out = self.drop(self.act(self.conv1(x)))
        out = self.drop(self.act(self.conv2(out)))
        return self.act(out + self.skip(x))


class TCNForecaster(nn.Module):
    """
    TCN for multivariate ECG forecasting.

    9 dilated causal conv layers: dilation = 1,2,4,...,256
    Receptive field = 2^9 = 512 ≥ INPUT_LEN ✅
    Every output timestep has access to the entire input window.

    Input  : (B, 12, 500)  channel-first
    Output : (B, HORIZON, 12)
    """
    def __init__(self, n_leads=12, n_filters=64, kernel_size=3,
                 n_layers=9, dropout=0.1, horizon=50):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads

        # Build dilated stack
        layers = []
        in_ch  = n_leads
        for i in range(n_layers):
            dilation = 2 ** i
            layers.append(TCNBlock(in_ch, n_filters, kernel_size, dilation, dropout))
            in_ch = n_filters
        self.tcn = nn.Sequential(*layers)

        # Output: last n_filters channels → HORIZON × n_leads
        # Use final HORIZON timesteps of the sequence output
        self.out = nn.Sequential(
            nn.Conv1d(n_filters, 64, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(64, n_leads, kernel_size=1),
        )

    def forward(self, x):
        # x: (B, 12, 500)
        feats = self.tcn(x)                # (B, n_filters, 500)
        out   = self.out(feats)            # (B, n_leads, 500)
        # Take last HORIZON timesteps as forecast
        out   = out[:, :, -self.horizon:]  # (B, n_leads, HORIZON)
        return out.permute(0, 2, 1)        # (B, HORIZON, n_leads)


tcn_model = TCNForecaster(
    n_leads=N_LEADS, n_filters=64, kernel_size=3,
    n_layers=9, dropout=0.1, horizon=HORIZON
).to(DEVICE)

n_tcn = sum(p.numel() for p in tcn_model.parameters() if p.requires_grad)

with torch.no_grad():
    d = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)
    o = tcn_model(d)
    print(f"TCN: {d.shape} → {o.shape}  ✅")
print(f"Params     : {n_tcn:,}")
print(f"Receptive field: {2**9} samples ≥ {INPUT_LEN} ✅")

AttributeError: 'CausalConv1d' object has no attribute 'weight'

### Cell 10 — Model E: WaveNet-Style Dilated CNN

In [ ]:
# ============================================================
# MODEL E: WAVENET-STYLE DILATED CNN
#
# WaveNet uses Gated Activation Units (not GELU) and skip connections
# summed across ALL layers — a richer feature aggregation than TCN.
#
# Key differences from TCN:
# - Gated activation: tanh(conv) * sigmoid(conv)  — multiplicative gate
# - Skip connections: every layer contributes to final output
# - 1×1 residual and skip projections
#
# Originally designed for audio waveform generation — ECG is also
# a continuous waveform with multi-scale temporal structure.
# ============================================================
class WaveNetBlock(nn.Module):
    """
    WaveNet residual block with gated activation and skip connection.
    Uses causal dilated convolutions (no future information leakage).
    """
    def __init__(self, n_filters, kernel_size, dilation):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        # Dilated causal conv — outputs 2× channels for gating
        self.dilated_conv = nn.Conv1d(
            n_filters, n_filters * 2, kernel_size,
            dilation=dilation, padding=pad
        )
        self.residual_conv = nn.Conv1d(n_filters, n_filters, 1)
        self.skip_conv     = nn.Conv1d(n_filters, n_filters, 1)

    def forward(self, x):
        residual = x
        h = self.dilated_conv(x)
        h = h[:, :, :x.size(2)]           # causal trim
        # Gated activation: tanh(left half) * sigmoid(right half)
        h_tanh  = torch.tanh(  h[:, :h.size(1)//2, :])
        h_sigm  = torch.sigmoid(h[:, h.size(1)//2:, :])
        h = h_tanh * h_sigm                # (B, n_filters, T)
        skip = self.skip_conv(h)
        out  = self.residual_conv(h) + residual
        return out, skip


class WaveNetForecaster(nn.Module):
    """
    WaveNet-style dilated CNN for ECG forecasting.

    Gated activation units capture non-linear waveform patterns.
    Skip connections from ALL layers are summed before decoding —
    each dilation level contributes to the final prediction.

    Input  : (B, 12, 500)  channel-first
    Output : (B, HORIZON, 12)
    """
    def __init__(self, n_leads=12, n_filters=64, kernel_size=3,
                 n_layers=9, dropout=0.1, horizon=50):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads

        self.input_conv = nn.Conv1d(n_leads, n_filters, 1)

        self.blocks = nn.ModuleList([
            WaveNetBlock(n_filters, kernel_size, dilation=2**i)
            for i in range(n_layers)
        ])

        self.drop = nn.Dropout(dropout)

        # Output from summed skips
        self.output_conv = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(n_filters, n_filters, 1),
            nn.ReLU(),
            nn.Conv1d(n_filters, n_leads, 1),
        )

    def forward(self, x):
        # x: (B, 12, 500)
        out  = self.input_conv(x)    # (B, n_filters, 500)
        skip_total = 0

        for block in self.blocks:
            out, skip = block(out)
            out        = self.drop(out)
            skip_total = skip_total + skip

        # Decode from skip sum
        out = self.output_conv(skip_total)   # (B, n_leads, 500)
        out = out[:, :, -self.horizon:]      # (B, n_leads, HORIZON)
        return out.permute(0, 2, 1)          # (B, HORIZON, n_leads)


wavenet_model = WaveNetForecaster(
    n_leads=N_LEADS, n_filters=64, kernel_size=3,
    n_layers=9, dropout=0.1, horizon=HORIZON
).to(DEVICE)

n_wave = sum(p.numel() for p in wavenet_model.parameters() if p.requires_grad)

with torch.no_grad():
    d = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)
    o = wavenet_model(d)
    print(f"WaveNet: {d.shape} → {o.shape}  ✅")
print(f"Params  : {n_wave:,}")

# Summary
print("\n" + "="*55)
print("  ALL 5 MODELS — PARAMETER COMPARISON")
print("="*55)
print(f"  A. Seq2Seq LSTM  : {n_lstm:>8,}")
print(f"  B. CNN-LSTM      : {n_cnn:>8,}")
print(f"  C. Transformer   : {n_trans:>8,}")
print(f"  D. TCN           : {n_tcn:>8,}")
print(f"  E. WaveNet       : {n_wave:>8,}")

### Cell 11 — Training Engine

In [ ]:
# ============================================================
# TRAINING ENGINE
#
# Loss: MSE — direct, no Huber smoothing that hides peaks
#       MSE penalises QRS amplitude errors more than Huber,
#       encouraging the model to predict peaks correctly.
#
# Teacher forcing schedule: start at 0.5, decay to 0.0
#   This helps seq2seq models bootstrap early training
#   then learn to generate independently.
# ============================================================

FIG_DIR  = os.path.join('..', 'reports', 'figures')
CKPT_DIR = os.path.join('..', 'reports', 'checkpoints')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

CRITERION = nn.MSELoss()   # direct MSE — no Huber smoothing

def get_tf_ratio(epoch, total_epochs, start=0.5):
    """Linear decay of teacher forcing ratio from start to 0."""    return max(0.0, start * (1 - epoch / total_epochs))

def train_epoch(model, loader, optimizer, epoch, total_epochs, is_seq2seq=False):
    model.train()
    total = 0.0
    tf    = get_tf_ratio(epoch, total_epochs) if is_seq2seq else 0.0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if is_seq2seq:
            pred = model(xb, teacher_forcing_ratio=tf, target=yb)
        else:
            pred = model(xb)

        loss = CRITERION(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * len(xb)
    return total / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, is_seq2seq=False):
    model.eval()
    total  = 0.0
    preds  = []
    labels = []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        pred   = model(xb)
        total += CRITERION(pred, yb).item() * len(xb)
        preds.append(pred.cpu().numpy())
        labels.append(yb.cpu().numpy())
    return (total / len(loader.dataset),
            np.concatenate(preds),
            np.concatenate(labels))

def train_model(model, tr_loader, vl_loader, model_name,
                n_epochs=80, lr=3e-4, patience=15, is_seq2seq=False):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # Warm restarts — helps escape local minima
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

    best_val   = float('inf')
    no_improve = 0
    ckpt       = os.path.join(CKPT_DIR, f'{model_name}_best.pt')
    history    = {'tr': [], 'vl': []}

    print(f"\n{'─'*70}")
    print(f"  Training: {model_name}  |  Teacher forcing: {is_seq2seq}")
    print(f"{'─'*70}")
    print(f"  {'Ep':>4}  {'Train MSE':>12}  {'Val MSE':>12}  {'Val RMSE':>10}  {'LR':>10}")
    print(f"  {'-'*60}")

    for ep in range(1, n_epochs+1):
        tr  = train_epoch(model, tr_loader, optimizer, ep, n_epochs, is_seq2seq)
        vl, preds, tgts = eval_epoch(model, vl_loader, is_seq2seq)
        scheduler.step(ep)

        history['tr'].append(tr)
        history['vl'].append(vl)

        best = vl < best_val
        if best:
            best_val   = vl
            no_improve = 0
            torch.save(model.state_dict(), ckpt)
        else:
            no_improve += 1

        lr_now = optimizer.param_groups[0]['lr']
        flag   = ' ★' if best else ''
        vl_rmse = np.sqrt(vl)
        print(f"  {ep:>4}  {tr:>12.6f}  {vl:>12.6f}  {vl_rmse:>10.4f}  {lr_now:>10.2e}{flag}")

        if no_improve >= patience:
            print(f"\n  Early stop at epoch {ep}")
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print(f"\n  Best val MSE: {best_val:.6f}  RMSE: {np.sqrt(best_val):.4f}  ✅")
    return history

print("Training engine ready ✅")
print(f"  Loss      : MSELoss (direct — no Huber smoothing)")
print(f"  Optimizer : AdamW")
print(f"  Scheduler : CosineAnnealingWarmRestarts (T_0=20)")
print(f"  Grad clip : 1.0")

### Cell 12 — Train All 5 Models

In [ ]:
# ============================================================
# TRAIN ALL 5 MODELS
# is_seq2seq=True → teacher forcing schedule applied
# CNN/TCN/WaveNet use channel-first loaders
# ============================================================
histories = {}

# A. Seq2Seq LSTM
histories['Seq2Seq-LSTM'] = train_model(
    lstm_model, lstm_tr, lstm_vl,
    'Seq2Seq-LSTM', n_epochs=80, lr=3e-4,
    patience=15, is_seq2seq=True
)

# B. CNN-LSTM
histories['CNN-LSTM'] = train_model(
    cnn_lstm_model, cnn_tr, cnn_vl,
    'CNN-LSTM', n_epochs=80, lr=3e-4,
    patience=15, is_seq2seq=True
)

# C. Transformer
histories['Transformer'] = train_model(
    transformer_model, lstm_tr, lstm_vl,
    'Transformer', n_epochs=80, lr=1e-4,
    patience=15, is_seq2seq=False
)

# D. TCN (channel-first, no teacher forcing)
histories['TCN'] = train_model(
    tcn_model, cnn_tr, cnn_vl,
    'TCN', n_epochs=80, lr=3e-4,
    patience=15, is_seq2seq=False
)

# E. WaveNet (channel-first, no teacher forcing)
histories['WaveNet'] = train_model(
    wavenet_model, cnn_tr, cnn_vl,
    'WaveNet', n_epochs=80, lr=3e-4,
    patience=15, is_seq2seq=False
)

print("\n✅ All 5 models trained")

### Cell 13 — Evaluation: All Metrics Including Pearson r

In [ ]:
# ============================================================
# COMPREHENSIVE EVALUATION
# MAE, RMSE, MAPE, Pearson r per lead + macro
# Pearson r measures WAVEFORM MORPHOLOGY PRESERVATION —
# the key metric to show the model isn't just predicting mean
# ============================================================

def compute_metrics(y_true, y_pred, lead_names):
    """MAE, RMSE, MAPE, Pearson r per lead + macro."""    results = {}
    maes, rmses, mapes, corrs = [], [], [], []

    for i, name in enumerate(lead_names):
        yt = y_true[:, :, i].flatten()
        yp = y_pred[:, :, i].flatten()

        mae  = mean_absolute_error(yt, yp)
        rmse = np.sqrt(mean_squared_error(yt, yp))
        mask = np.abs(yt) > 0.05
        mape = np.mean(np.abs((yt[mask]-yp[mask])/yt[mask]))*100 if mask.sum()>0 else np.nan
        r, _ = pearsonr(yt, yp)

        results[name] = {'MAE':mae,'RMSE':rmse,'MAPE':mape,'Pearson_r':r}
        maes.append(mae); rmses.append(rmse)
        if not np.isnan(mape): mapes.append(mape)
        if not np.isnan(r):    corrs.append(r)

    results['MACRO'] = {
        'MAE':  np.mean(maes),
        'RMSE': np.mean(rmses),
        'MAPE': np.mean(mapes) if mapes else np.nan,
        'Pearson_r': np.mean(corrs) if corrs else np.nan,
    }
    return results

# Models and their loaders
models_eval = {
    'Seq2Seq-LSTM': (lstm_model, lstm_te),
    'CNN-LSTM'    : (cnn_lstm_model, cnn_te),
    'Transformer' : (transformer_model, lstm_te),
    'TCN'         : (tcn_model, cnn_te),
    'WaveNet'     : (wavenet_model, cnn_te),
}

all_results = {}
all_preds   = {}

for mname, (model, loader) in models_eval.items():
    vl, preds, tgts = eval_epoch(model, loader)
    all_preds[mname]   = preds
    all_results[mname] = compute_metrics(tgts, preds, LEAD_NAMES)

# Print full results table
print("=" * 75)
print("  TEST SET RESULTS — ALL 5 MODELS + PERSISTENCE BASELINE")
print("=" * 75)
print(f"  {'Model':<16} {'MAE':>7} {'RMSE':>7} {'MAPE%':>7} {'Pearson r':>10}")
print(f"  {'-'*55}")
print(f"  {'Persistence':<16} {persist_mae:>7.4f} {persist_rmse:>7.4f} {'—':>7} {persist_corr:>10.4f}")
print(f"  {'-'*55}")
for mname in all_results:
    m = all_results[mname]['MACRO']
    imp = '✅' if m['RMSE'] < persist_rmse else '❌'
    print(f"  {mname:<16} {m['MAE']:>7.4f} {m['RMSE']:>7.4f} {m['MAPE']:>6.1f}% {m['Pearson_r']:>10.4f}  {imp}")

print()
best_name = min(all_results, key=lambda n: all_results[n]['MACRO']['RMSE'])
best_corr = max(all_results, key=lambda n: all_results[n]['MACRO']['Pearson_r'])
print(f"  🏆 Best RMSE      : {best_name}")
print(f"  🏆 Best Pearson r : {best_corr}")

In [ ]:
# ── Per-lead detailed table for best model ────────────────────
print("\n" + "="*60)
print(f"  PER-LEAD RESULTS — {best_name}")
print("="*60)
print(f"  {'Lead':<6} {'MAE':>7} {'RMSE':>7} {'MAPE%':>7} {'Pearson r':>10}")
print(f"  {'-'*45}")
for lead in LEAD_NAMES:
    m = all_results[best_name][lead]
    print(f"  {lead:<6} {m['MAE']:>7.4f} {m['RMSE']:>7.4f} {m['MAPE']:>6.1f}%  {m['Pearson_r']:>10.4f}")
print(f"  {'-'*45}")
m = all_results[best_name]['MACRO']
print(f"  {'MACRO':<6} {m['MAE']:>7.4f} {m['RMSE']:>7.4f} {m['MAPE']:>6.1f}%  {m['Pearson_r']:>10.4f}")

### Cell 15 — Training Loss Curves

In [ ]:
colors5 = {
    'Seq2Seq-LSTM': '#0ea5e9',
    'CNN-LSTM'    : '#10b981',
    'Transformer' : '#f472b6',
    'TCN'         : '#f59e0b',
    'WaveNet'     : '#8b5cf6',
}

fig, axes = plt.subplots(1, 5, figsize=(24, 4))
for ax, (name, hist) in zip(axes, histories.items()):
    ep = range(1, len(hist['tr'])+1)
    ax.plot(ep, hist['tr'], color=colors5[name], lw=1.4, label='Train MSE')
    ax.plot(ep, hist['vl'], color=colors5[name], lw=1.4, ls='--', alpha=0.7, label='Val MSE')
    best_ep = int(np.argmin(hist['vl'])) + 1
    ax.axvline(best_ep, color='gold', ls=':', lw=1.5)
    ax.set_title(name, fontsize=10, fontweight='bold', color=colors5[name])
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
    m = all_results[name]['MACRO']
    ax.set_xlabel(f"Best RMSE={m['RMSE']:.4f}  r={m['Pearson_r']:.3f}", fontsize=8)

fig.suptitle('Training & Validation MSE Loss — All 5 Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show(); print("✅ Saved → reports/figures/training_curves.png")

### Cell 16 — Predicted vs Actual Overlay

In [ ]:
# ============================================================
# PREDICTED VS ACTUAL — 3 TEST WINDOWS, ALL 5 MODELS
# This is the key diagnostic plot.
# A working model should show:
# - Continuation of rhythm
# - Recognisable QRS peaks
# - Phase continuity
# ============================================================
t_in  = np.arange(INPUT_LEN) / FS
t_out = np.arange(INPUT_LEN, INPUT_LEN + HORIZON) / FS

fig, axes = plt.subplots(3, 1, figsize=(20, 12))

for s_idx, ax in enumerate(axes):
    # Input context — Lead II
    ax.plot(t_in, X_test[s_idx, :, 1],
            color='#94a3b8', lw=0.9, alpha=0.8, label='Input (5s)')
    # Ground truth
    ax.plot(t_out, y_test[s_idx, :, 1],
            color='white', lw=2.5, label='Ground truth', zorder=5)

    for mname, col in colors5.items():
        yp = all_preds[mname][s_idx, :, 1]
        ax.plot(t_out, yp, color=col, lw=1.3, ls='--',
                alpha=0.9, label=mname)

    ax.axvline(INPUT_LEN/FS, color='#facc15', ls='--', lw=1.5, alpha=0.7)
    ax.fill_betweenx([-4,4], INPUT_LEN/FS, (INPUT_LEN+HORIZON)/FS,
                     alpha=0.06, color='white')
    ax.set_title(f'Test window #{s_idx} — Lead II', fontweight='bold')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('mV (normalised)')
    ax.legend(fontsize=7, loc='upper left', ncol=4); ax.grid(alpha=0.25)
    ax.set_xlim([0, (INPUT_LEN+HORIZON)/FS])
    ax.set_ylim([-4, 4])

fig.suptitle('ECG Forecasting — Predicted vs Actual (Lead II)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'forecast_overlay.png'), dpi=150, bbox_inches='tight')
plt.show(); print("✅ Saved → reports/figures/forecast_overlay.png")

### Cell 17 — Per-Lead RMSE + Pearson r Bar Charts

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(28, 9))
x_pos = np.arange(len(LEAD_NAMES))
w = 0.7

for col_i, (mname, col) in enumerate(colors5.items()):
    # RMSE
    ax = axes[0, col_i]
    rmses = [all_results[mname][l]['RMSE'] for l in LEAD_NAMES]
    macro_rmse = all_results[mname]['MACRO']['RMSE']
    bars = ax.bar(x_pos, rmses, w, color=col, alpha=0.85)
    ax.axhline(macro_rmse, color='#facc15', ls='--', lw=1.5,
               label=f'Macro={macro_rmse:.3f}')
    ax.axhline(persist_rmse, color='red', ls=':', lw=1, alpha=0.6, label='Persist')
    ax.set_xticks(x_pos); ax.set_xticklabels(LEAD_NAMES, rotation=45, fontsize=7)
    ax.set_title(f'{mname}', fontweight='bold', color=col, fontsize=9)
    ax.set_ylabel('RMSE'); ax.legend(fontsize=6); ax.grid(axis='y', alpha=0.3)

    # Pearson r
    ax2 = axes[1, col_i]
    corrs = [all_results[mname][l]['Pearson_r'] for l in LEAD_NAMES]
    macro_r = all_results[mname]['MACRO']['Pearson_r']
    bars2 = ax2.bar(x_pos, corrs, w, color=col, alpha=0.85)
    ax2.axhline(macro_r, color='#facc15', ls='--', lw=1.5, label=f'Macro r={macro_r:.3f}')
    ax2.axhline(0, color='red', ls=':', lw=1, alpha=0.5, label='r=0 (chance)')
    ax2.set_xticks(x_pos); ax2.set_xticklabels(LEAD_NAMES, rotation=45, fontsize=7)
    ax2.set_title(f'Pearson r — {mname}', fontsize=9)
    ax2.set_ylabel('Pearson r'); ax2.legend(fontsize=6); ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim([-0.1, 1.0])

fig.suptitle('Per-Lead RMSE (top) and Pearson r (bottom) — All 5 Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'per_lead_metrics.png'), dpi=150, bbox_inches='tight')
plt.show(); print("✅ Saved → reports/figures/per_lead_metrics.png")

### Cell 18 — Final Sanity Check

In [ ]:
# ============================================================
# FINAL SANITY CHECK — proves models learned ECG dynamics
# A model that just predicts the mean will have:
#   - Pearson r ≈ 0 (no correlation with actual waveform)
#   - RMSE > persistence baseline
# A model that learned ECG dynamics will have:
#   - Pearson r > 0.3  (positive correlation)
#   - RMSE < persistence baseline
# ============================================================
print("=" * 65)
print("  FINAL SANITY CHECK")
print("=" * 65)
errors = []

def chk(cond, label):
    sym = "✅" if cond else "❌"
    print(f"  {sym}  {label}")
    if not cond: errors.append(label)

# Shape checks
for mname, preds in all_preds.items():
    chk(preds.shape == y_test.shape,
        f"{mname}: output shape {preds.shape} == y_test {y_test.shape}")

print()
# Beat persistence
for mname, preds in all_preds.items():
    r     = all_results[mname]['MACRO']
    beats = r['RMSE'] < persist_rmse
    chk(beats, f"{mname}: RMSE {r['RMSE']:.4f} < persistence {persist_rmse:.4f}")

print()
# Pearson correlation > 0 (not mean collapse)
for mname in all_results:
    r  = all_results[mname]['MACRO']['Pearson_r']
    chk(r > 0.1, f"{mname}: Pearson r={r:.4f} > 0.1 (not mean collapse)")

print()
# Predictions are not constant (std > 0.01)
for mname, preds in all_preds.items():
    std = preds.std()
    chk(std > 0.05, f"{mname}: prediction std={std:.4f} > 0.05 (not flat line)")

print()
print("=" * 65)
if errors:
    print(f"  ⚠️  {len(errors)} checks failed — review architecture or training")
    for e in errors: print(f"     • {e}")
else:
    print("  ✅ ALL CHECKS PASSED")
    print("  Models are learning ECG dynamics, not mean collapse")
print("=" * 65)